# BAN404 — Python Recipe Book
### Exam preparation: May 19, 2026

Every method in the compendium, one clean code block at a time.  
**Pattern throughout:** `split → (scale) → fit → predict → evaluate`

Run each cell in order — all examples use synthetic or built-in datasets so nothing external is needed.


## 0 · Global Imports & Synthetic Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

# --- Model selection / preprocessing ---
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     KFold, LeaveOneOut, GridSearchCV)
from sklearn.preprocessing  import StandardScaler, PolynomialFeatures, SplineTransformer
from sklearn.pipeline        import Pipeline

# --- Regression ---
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.decomposition import PCA

# --- Classification ---
from sklearn.linear_model            import LogisticRegression
from sklearn.discriminant_analysis   import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis   import QuadraticDiscriminantAnalysis as QDA
from sklearn.naive_bayes             import GaussianNB
from sklearn.neighbors               import KNeighborsClassifier, KNeighborsRegressor

# --- Trees & Ensembles ---
from sklearn.tree     import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                               GradientBoostingClassifier,
                               BaggingRegressor,  RandomForestRegressor,
                               GradientBoostingRegressor)

# --- SVM ---
from sklearn.svm import SVC, SVR

# --- Neural Networks ---
from sklearn.neural_network import MLPClassifier, MLPRegressor

# --- Clustering ---
from sklearn.cluster import KMeans, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# --- Metrics ---
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_auc_score, roc_curve,
                              mean_squared_error, r2_score, silhouette_score)

# ── Synthetic datasets used throughout ──────────────────────────────────────
from sklearn.datasets import make_regression, make_classification

np.random.seed(42)

# Regression dataset: 200 obs, 5 predictors, 2 informative
X_reg, y_reg = make_regression(n_samples=200, n_features=5, noise=30, random_state=42)

# Classification dataset: 300 obs, 5 predictors, 2 classes
X_clf, y_clf = make_classification(n_samples=300, n_features=5, n_informative=3,
                                    random_state=42)

print("Regression dataset :", X_reg.shape, "→", y_reg.shape)
print("Classification dataset:", X_clf.shape, "→", y_clf.shape)


---
## 1 · Chapters 1–2 — Splitting Data & Evaluating Models

The most fundamental operation: split data **before** any fitting.  
Never let test data influence preprocessing or model selection.


In [ ]:
# ── Train / Test split ──────────────────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    X_reg, y_reg,
    test_size=0.2,    # 20 % test, 80 % train
    random_state=42
)
print(f"Train: {X_tr.shape}  |  Test: {X_te.shape}")

# ── Fit a simple model and evaluate on test set ─────────────────────────────
model = LinearRegression()
model.fit(X_tr, y_tr)          # learn from training data only

y_pred = model.predict(X_te)   # predict on test data

mse  = mean_squared_error(y_te, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_te, y_pred)

print(f"Test MSE : {mse:.2f}")
print(f"Test RMSE: {rmse:.2f}")
print(f"Test R²  : {r2:.4f}")


---
## 2 · Chapter 3 — Linear Regression


In [ ]:
# ── OLS via sklearn ──────────────────────────────────────────────────────────
ols = LinearRegression()
ols.fit(X_tr, y_tr)

print("Intercept :", ols.intercept_)
print("Coefficients:", ols.coef_)
print("Test R²   :", r2_score(y_te, ols.predict(X_te)).round(4))


In [ ]:
# ── OLS with full summary (t-stats, p-values, CIs) via statsmodels ──────────
import statsmodels.api as sm

X_tr_sm = sm.add_constant(X_tr)   # statsmodels needs intercept added manually
sm_model = sm.OLS(y_tr, X_tr_sm).fit()

print(sm_model.summary())
# Key columns: coef, std err, t, P>|t|, [0.025, 0.975]


In [ ]:
# ── Prediction interval vs Confidence interval ───────────────────────────────
# statsmodels gives both for a new observation
X_new = sm.add_constant(X_te[:3])   # first 3 test observations

pred_ci = sm_model.get_prediction(X_new).summary_frame(alpha=0.05)
print(pred_ci[['mean', 'mean_ci_lower', 'mean_ci_upper',   # CI for E[Y|x]
                'obs_ci_lower', 'obs_ci_upper']])           # PI for individual Y
# PI is always wider than CI — includes irreducible error Var(ε)


---
## 3 · Chapter 4 — Classification

Common pattern for all classifiers:
```
model.fit(X_tr, y_tr)
y_pred  = model.predict(X_te)          # hard class labels
y_prob  = model.predict_proba(X_te)    # class probabilities
```


In [ ]:
# ── Split classification data ────────────────────────────────────────────────
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)


In [ ]:
# ── Logistic Regression ──────────────────────────────────────────────────────
lr = LogisticRegression(max_iter=1000)
lr.fit(Xc_tr, yc_tr)

y_pred = lr.predict(Xc_te)            # class 0 or 1
y_prob = lr.predict_proba(Xc_te)[:,1] # P(Y=1 | X) for ROC

print(classification_report(yc_te, y_pred))
print("AUC:", roc_auc_score(yc_te, y_prob).round(4))

# Log-odds coefficients — positive means predictor raises P(Y=1)
print("Log-odds coefs:", lr.coef_)


In [ ]:
# ── Custom threshold (cost-sensitive) ────────────────────────────────────────
# Default threshold is 0.5; change if FN and FP have different costs
threshold = 0.3   # lower → catch more positives (reduce FN at cost of more FP)
y_pred_custom = (y_prob >= threshold).astype(int)
print(confusion_matrix(yc_te, y_pred_custom))
# Rows = actual, Cols = predicted: [[TN FP], [FN TP]]


In [ ]:
# ── ROC Curve ────────────────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(yc_te, y_prob)
auc = roc_auc_score(yc_te, y_prob)

plt.figure(figsize=(5,4))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0,1],[0,1],'--', color='grey', label='Random')
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Logistic Regression"); plt.legend(); plt.tight_layout()
plt.show()


In [ ]:
# ── LDA ──────────────────────────────────────────────────────────────────────
lda = LDA()
lda.fit(Xc_tr, yc_tr)
print("LDA accuracy:", lda.score(Xc_te, yc_te).round(4))
# LDA assumes Gaussian features with shared covariance matrix Σ


In [ ]:
# ── QDA ──────────────────────────────────────────────────────────────────────
qda = QDA()
qda.fit(Xc_tr, yc_tr)
print("QDA accuracy:", qda.score(Xc_te, yc_te).round(4))
# QDA allows class-specific covariance matrices Σ_k — more flexible, more parameters


In [ ]:
# ── Naive Bayes ──────────────────────────────────────────────────────────────
nb = GaussianNB()
nb.fit(Xc_tr, yc_tr)
print("Naive Bayes accuracy:", nb.score(Xc_te, yc_te).round(4))
# Assumes all features are conditionally independent given the class


In [ ]:
# ── K-Nearest Neighbours (KNN) ───────────────────────────────────────────────
# Scale first — KNN uses Euclidean distance, scale matters!
scaler = StandardScaler()
Xc_tr_sc = scaler.fit_transform(Xc_tr)   # fit on train, transform train
Xc_te_sc = scaler.transform(Xc_te)       # transform test with SAME scaler

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(Xc_tr_sc, yc_tr)
print("KNN (k=5) accuracy:", knn.score(Xc_te_sc, yc_te).round(4))

# Choose optimal K via cross-validation
cv_scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k),
                              Xc_tr_sc, yc_tr, cv=5).mean()
             for k in range(1, 21)]
best_k = np.argmax(cv_scores) + 1
print(f"Best K: {best_k}  (CV accuracy: {max(cv_scores):.4f})")


---
## 4 · Chapter 5 — Resampling Methods


In [ ]:
# ── k-Fold Cross-Validation ──────────────────────────────────────────────────
model = LinearRegression()

# cross_val_score: splits, fits, and evaluates in one call
cv_scores = cross_val_score(model, X_reg, y_reg,
                             cv=5,                    # 5 folds
                             scoring='neg_mean_squared_error')

cv_mse = -cv_scores                # sklearn returns negative MSE
print("5-fold CV MSE per fold:", cv_mse.round(2))
print("Mean CV MSE:", cv_mse.mean().round(2))
print("Std  CV MSE:", cv_mse.std().round(2))


In [ ]:
# ── Leave-One-Out CV (LOOCV) ─────────────────────────────────────────────────
loo = LeaveOneOut()
loocv_scores = cross_val_score(model, X_reg, y_reg,
                                cv=loo,
                                scoring='neg_mean_squared_error')

print(f"LOOCV MSE: {(-loocv_scores).mean():.2f}")
# LOOCV = n separate fits, one per observation. Expensive but low bias.
# For OLS, hat matrix shortcut exists: CV_LOOCV = (1/n) Σ (yᵢ - ŷᵢ)² / (1 - hᵢᵢ)²


In [ ]:
# ── Bootstrap ────────────────────────────────────────────────────────────────
# Estimate sampling distribution of a statistic (e.g. median)
n = len(y_reg)
B = 1000   # number of bootstrap samples
boot_medians = []

for _ in range(B):
    idx = np.random.choice(n, size=n, replace=True)   # sample WITH replacement
    boot_medians.append(np.median(y_reg[idx]))

boot_medians = np.array(boot_medians)
print(f"Bootstrap estimate of median : {boot_medians.mean():.4f}")
print(f"Bootstrap SE of median       : {boot_medians.std():.4f}")
print(f"95% CI (percentile method)   : [{np.percentile(boot_medians, 2.5):.3f}, "
      f"{np.percentile(boot_medians, 97.5):.3f}]")
# ~36.8% of observations are out-of-bag (OOB) per bootstrap sample


---
## 5 · Chapter 6 — Regularisation (Ridge, Lasso, PCR)


In [ ]:
# ── Always scale before regularisation ──────────────────────────────────────
scaler = StandardScaler()
Xr_tr_sc = scaler.fit_transform(X_tr)
Xr_te_sc = scaler.transform(X_te)

# ── Ridge (L2) ───────────────────────────────────────────────────────────────
ridge = Ridge(alpha=10.0)   # alpha = λ (regularisation strength)
ridge.fit(Xr_tr_sc, y_tr)
print("Ridge coefs:", ridge.coef_.round(2))
print("Ridge R²   :", r2_score(y_te, ridge.predict(Xr_te_sc)).round(4))
# Ridge shrinks all coefs toward 0, never exactly 0


In [ ]:
# ── Lasso (L1) ───────────────────────────────────────────────────────────────
lasso = Lasso(alpha=10.0)
lasso.fit(Xr_tr_sc, y_tr)
print("Lasso coefs:", lasso.coef_.round(2))   # some will be exactly 0 → feature selection
print("Lasso R²   :", r2_score(y_te, lasso.predict(Xr_te_sc)).round(4))


In [ ]:
# ── Choose λ via cross-validation ────────────────────────────────────────────
from sklearn.linear_model import RidgeCV, LassoCV

# RidgeCV and LassoCV do CV internally across a range of alphas
alphas = np.logspace(-3, 3, 100)   # 100 values from 0.001 to 1000

ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(Xr_tr_sc, y_tr)
print("Best Ridge alpha:", ridge_cv.alpha_)

lasso_cv = LassoCV(alphas=alphas, cv=5, max_iter=10000)
lasso_cv.fit(Xr_tr_sc, y_tr)
print("Best Lasso alpha:", lasso_cv.alpha_)


In [ ]:
# ── Regularisation path: coefficients vs log(λ) ──────────────────────────────
alphas_path = np.logspace(-2, 4, 50)
ridge_coefs = [Ridge(alpha=a).fit(Xr_tr_sc, y_tr).coef_ for a in alphas_path]

plt.figure(figsize=(7,4))
plt.plot(np.log10(alphas_path), ridge_coefs)
plt.axvline(np.log10(ridge_cv.alpha_), ls='--', color='k', label='CV best λ')
plt.xlabel("log₁₀(λ)"); plt.ylabel("Coefficient value")
plt.title("Ridge regularisation path"); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Principal Components Regression (PCR) ────────────────────────────────────
# PCR = PCA for dimensionality reduction, then OLS on the PCs
# Use Pipeline to chain steps cleanly
pcr = Pipeline([
    ('scale', StandardScaler()),     # step 1: standardise
    ('pca',   PCA(n_components=3)),  # step 2: keep first 3 PCs
    ('ols',   LinearRegression())    # step 3: OLS on PC scores
])
pcr.fit(X_tr, y_tr)
print("PCR (3 PCs) R²:", r2_score(y_te, pcr.predict(X_te)).round(4))

# Choose n_components via CV
cv_r2 = [cross_val_score(Pipeline([('sc', StandardScaler()),
                                     ('pca', PCA(n_components=m)),
                                     ('ols', LinearRegression())]),
                          X_tr, y_tr, cv=5, scoring='r2').mean()
          for m in range(1, X_tr.shape[1]+1)]
print("CV R² per n_components:", [round(v,3) for v in cv_r2])


---
## 6 · Chapter 7 — Moving Beyond Linearity


In [ ]:
# Use 1D data for easy visualisation
np.random.seed(42)
x1d = np.sort(np.random.uniform(-3, 3, 150))
y1d = np.sin(x1d) * 3 + np.random.normal(0, 0.5, 150)
X1d = x1d.reshape(-1, 1)

x_plot = np.linspace(-3, 3, 300).reshape(-1, 1)   # dense grid for plotting

plt.scatter(x1d, y1d, s=15, alpha=0.5, label='data')
plt.title("1D dataset — sin curve + noise"); plt.xlabel("x"); plt.ylabel("y")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Polynomial Regression ────────────────────────────────────────────────────
poly = Pipeline([
    ('poly',  PolynomialFeatures(degree=4, include_bias=False)),  # x, x², x³, x⁴
    ('scale', StandardScaler()),
    ('ols',   LinearRegression())
])
poly.fit(X1d, y1d)

plt.scatter(x1d, y1d, s=15, alpha=0.5)
plt.plot(x_plot, poly.predict(x_plot), color='red', label='Degree-4 polynomial')
plt.title("Polynomial Regression (degree=4)"); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── B-Spline Regression (SplineTransformer) ──────────────────────────────────
spline = Pipeline([
    ('spline', SplineTransformer(n_knots=6, degree=3)),  # cubic spline, 6 knots
    ('ols',    LinearRegression())
])
spline.fit(X1d, y1d)

plt.scatter(x1d, y1d, s=15, alpha=0.5)
plt.plot(x_plot, spline.predict(x_plot), color='green', label='Cubic spline (6 knots)')
plt.title("Spline Regression"); plt.legend(); plt.tight_layout(); plt.show()
# Natural splines: use extrapolation_kind='linear' in SplineTransformer (sklearn ≥1.2)


In [ ]:
# ── Generalised Additive Model (GAM) — manual backfitting intuition ───────────
# sklearn has no native GAM, but a GAM on tabular data can be approximated by
# fitting spline transforms on each feature independently and summing.
# In practice, use the 'pygam' library (pip install pygam) for real GAMs.

# Example: GAM-like pipeline with spline features for each predictor
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

# For each of the 5 predictors, apply a cubic spline transformation
spline_per_col = ColumnTransformer(
    [('sp_' + str(j), SplineTransformer(n_knots=4, degree=3), [j])
     for j in range(X_tr.shape[1])]
)
gam_approx = Pipeline([('splines', spline_per_col), ('ols', LinearRegression())])
gam_approx.fit(X_tr, y_tr)
print("GAM-approx test R²:", r2_score(y_te, gam_approx.predict(X_te)).round(4))


---
## 7 · Chapter 8 — Tree-Based Methods


In [ ]:
# ── Decision Tree (Classification) ───────────────────────────────────────────
tree = DecisionTreeClassifier(max_depth=4,   # controls tree complexity
                               random_state=42)
tree.fit(Xc_tr, yc_tr)
print("Tree accuracy:", tree.score(Xc_te, yc_te).round(4))

# ── Cost-complexity pruning: choose alpha via CV ──────────────────────────────
path = tree.cost_complexity_pruning_path(Xc_tr, yc_tr)
alphas = path.ccp_alphas[::3]   # subsample alphas for speed

cv_acc = [cross_val_score(DecisionTreeClassifier(ccp_alpha=a, random_state=42),
                           Xc_tr, yc_tr, cv=5).mean()
          for a in alphas]
best_alpha = alphas[np.argmax(cv_acc)]
print(f"Best ccp_alpha: {best_alpha:.5f}")

pruned_tree = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42)
pruned_tree.fit(Xc_tr, yc_tr)
print("Pruned tree accuracy:", pruned_tree.score(Xc_te, yc_te).round(4))


In [ ]:
# ── Bagging ───────────────────────────────────────────────────────────────────
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=200,     # B bootstrap trees
    oob_score=True,       # free OOB error estimate
    random_state=42
)
bag.fit(Xc_tr, yc_tr)
print("Bagging test accuracy:", bag.score(Xc_te, yc_te).round(4))
print("Bagging OOB  accuracy:", bag.oob_score_.round(4))
# OOB ≈ LOOCV estimate — no need for a separate validation set


In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=500,         # B trees
    max_features='sqrt',      # m = sqrt(p) predictors tried at each split (classification default)
    oob_score=True,
    random_state=42
)
rf.fit(Xc_tr, yc_tr)
print("RF test accuracy:", rf.score(Xc_te, yc_te).round(4))
print("RF OOB  accuracy:", rf.oob_score_.round(4))

# Variable importance (mean decrease in Gini impurity)
importances = pd.Series(rf.feature_importances_,
                         index=[f"X{i}" for i in range(Xc_tr.shape[1])])
importances.sort_values().plot.barh(title="Feature Importance (RF)")
plt.tight_layout(); plt.show()


In [ ]:
# ── Gradient Boosting ─────────────────────────────────────────────────────────
gb = GradientBoostingClassifier(
    n_estimators=300,     # B (number of trees / boosting rounds)
    learning_rate=0.05,   # ν (shrinkage) — small values need more trees
    max_depth=3,          # d (interaction depth) — usually 1–5
    subsample=0.8,        # stochastic GB: use 80% of data per tree
    random_state=42
)
gb.fit(Xc_tr, yc_tr)
print("GB test accuracy:", gb.score(Xc_te, yc_te).round(4))

# Variable importance
imp = pd.Series(gb.feature_importances_,
                 index=[f"X{i}" for i in range(Xc_tr.shape[1])])
imp.sort_values().plot.barh(title="Feature Importance (GB)")
plt.tight_layout(); plt.show()


In [ ]:
# ── Regression Trees (same API, different estimator) ─────────────────────────
tree_reg = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_reg.fit(X_tr, y_tr)
print("Regression tree test R²:", r2_score(y_te, tree_reg.predict(X_te)).round(4))

rf_reg = RandomForestRegressor(n_estimators=300, oob_score=True, random_state=42)
rf_reg.fit(X_tr, y_tr)
print("Random forest   test R²:", r2_score(y_te, rf_reg.predict(X_te)).round(4))

gb_reg = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                    max_depth=3, random_state=42)
gb_reg.fit(X_tr, y_tr)
print("Gradient boost  test R²:", r2_score(y_te, gb_reg.predict(X_te)).round(4))


---
## 8 · Chapter 9 — Support Vector Machines

**⚠️ sklearn convention:** `C` is the *inverse* of the textbook's budget.  
Large `C` → strict margin (few violations) → high variance.  
Small `C` → soft margin (many violations) → high bias.  
This is the **opposite** of the textbook's description.


In [ ]:
# Scale first — SVM is distance-based
scaler = StandardScaler()
Xs_tr = scaler.fit_transform(Xc_tr)
Xs_te = scaler.transform(Xc_te)


In [ ]:
# ── Linear SVC ───────────────────────────────────────────────────────────────
svc_lin = SVC(kernel='linear',
               C=1.0,          # regularisation (large C = strict margin)
               probability=True)   # needed for predict_proba / ROC
svc_lin.fit(Xs_tr, yc_tr)
print("Linear SVC accuracy:", svc_lin.score(Xs_te, yc_te).round(4))
print("Number of support vectors:", svc_lin.n_support_)


In [ ]:
# ── RBF (Gaussian) Kernel SVM ────────────────────────────────────────────────
svc_rbf = SVC(kernel='rbf',
               C=1.0,     # margin budget (sklearn convention)
               gamma=0.5, # 1/(2σ²): large γ → local, complex boundary
               probability=True)
svc_rbf.fit(Xs_tr, yc_tr)
print("RBF SVC accuracy:", svc_rbf.score(Xs_te, yc_te).round(4))


In [ ]:
# ── Tune C and gamma with GridSearchCV ───────────────────────────────────────
param_grid = {
    'C':     [0.1, 1, 10, 100],
    'gamma': [0.01, 0.1, 1, 'scale']
}
grid = GridSearchCV(SVC(kernel='rbf', probability=True),
                    param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(Xs_tr, yc_tr)

print("Best params:", grid.best_params_)
print("CV accuracy:", grid.best_score_.round(4))
print("Test accuracy:", grid.score(Xs_te, yc_te).round(4))


---
## 9 · Chapter 10 — Neural Networks (MLP)


In [ ]:
# Always scale for neural networks
scaler = StandardScaler()
Xn_tr = scaler.fit_transform(Xc_tr)
Xn_te = scaler.transform(Xc_te)


In [ ]:
# ── MLPClassifier ────────────────────────────────────────────────────────────
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),  # 2 hidden layers: 64 and 32 nodes
    activation='relu',            # ReLU in hidden layers
    solver='adam',                # Adam optimiser
    alpha=0.001,                  # L2 weight decay (lambda)
    batch_size=32,                # mini-batch size
    max_iter=200,                 # max epochs
    early_stopping=True,          # stop if val loss stops improving
    validation_fraction=0.1,      # 10% of train used for early stopping
    n_iter_no_change=10,          # patience
    random_state=42
)
mlp.fit(Xn_tr, yc_tr)
print("MLP accuracy:", mlp.score(Xn_te, yc_te).round(4))
print("Stopped at epoch:", mlp.n_iter_)


In [ ]:
# ── Plot training and validation loss curves ─────────────────────────────────
plt.figure(figsize=(6,4))
plt.plot(mlp.loss_curve_, label='Training loss')
if hasattr(mlp, 'validation_scores_'):
    plt.plot([-v for v in mlp.validation_scores_], label='Validation loss')
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.title("MLP Training Curve"); plt.legend(); plt.tight_layout(); plt.show()
# If val loss rises while train loss falls → overfitting (early stopping kicks in)


In [ ]:
# ── MLPRegressor ─────────────────────────────────────────────────────────────
scaler_r = StandardScaler()
Xnr_tr = scaler_r.fit_transform(X_tr)
Xnr_te = scaler_r.transform(X_te)

mlp_reg = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    max_iter=500,
    early_stopping=True,
    random_state=42
)
mlp_reg.fit(Xnr_tr, y_tr)
print("MLP Regression R²:", r2_score(y_te, mlp_reg.predict(Xnr_te)).round(4))


---
## 10 · Chapter 11 — Unsupervised Learning


In [ ]:
# Use the Iris dataset for unsupervised examples (4 features, known 3-class structure)
from sklearn.datasets import load_iris
iris = load_iris()
X_iris = iris.data          # 150 × 4
y_iris = iris.target        # true labels (only for validation, not used in fitting)


In [ ]:
# ── PCA ──────────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_iris)  # mandatory: standardise before PCA

pca = PCA()           # fit all components first to inspect variance
pca.fit(X_sc)

# Proportion of variance explained (PVE) per component
pve = pca.explained_variance_ratio_
print("PVE per component:", pve.round(4))
print("Cumulative PVE   :", pve.cumsum().round(4))


In [ ]:
# ── Scree plot ───────────────────────────────────────────────────────────────
plt.figure(figsize=(6,4))
plt.bar(range(1, len(pve)+1), pve, label='Individual PVE')
plt.plot(range(1, len(pve)+1), pve.cumsum(), 'o-', color='red', label='Cumulative PVE')
plt.axhline(0.9, ls='--', color='grey', label='90% threshold')
plt.xlabel("Principal Component"); plt.ylabel("Proportion of Variance Explained")
plt.title("Scree Plot"); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Project onto first 2 PCs and plot (biplot-like) ─────────────────────────
pca2 = PCA(n_components=2)
Z = pca2.fit_transform(X_sc)   # score matrix: n × 2

plt.figure(figsize=(6,5))
for k, name in enumerate(iris.target_names):
    mask = y_iris == k
    plt.scatter(Z[mask, 0], Z[mask, 1], label=name, s=30)

# Overlay loading vectors (arrows)
loadings = pca2.components_.T  # shape: (4, 2)
scale = 2.5
for i, feat in enumerate(iris.feature_names):
    plt.arrow(0, 0, loadings[i,0]*scale, loadings[i,1]*scale,
              head_width=0.08, color='black')
    plt.text(loadings[i,0]*scale*1.1, loadings[i,1]*scale*1.1, feat, fontsize=8)

plt.xlabel(f"PC1 ({pve[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pve[1]*100:.1f}% var)")
plt.title("Biplot — Iris (first 2 PCs)"); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── K-Means Clustering ───────────────────────────────────────────────────────
# Scale first (K-means uses Euclidean distance)
X_km = StandardScaler().fit_transform(X_iris)

# Elbow method: plot TWCSS (inertia) vs K
inertias = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_km)
    inertias.append(km.inertia_)   # total within-cluster SS

plt.figure(figsize=(6,4))
plt.plot(K_range, inertias, 'o-')
plt.xlabel("K"); plt.ylabel("Total within-cluster SS (inertia)")
plt.title("Elbow Method"); plt.tight_layout(); plt.show()


In [ ]:
# ── K-Means with K=3 ─────────────────────────────────────────────────────────
km3 = KMeans(n_clusters=3, n_init=20,   # 20 random starts, keep best
              random_state=42)
km3.fit(X_km)
labels_km = km3.labels_

print("Cluster sizes:", np.bincount(labels_km))
print("Silhouette score:", silhouette_score(X_km, labels_km).round(4))
# Silhouette ∈ [-1,1]; closer to 1 = better separation

# Compare to true labels (for validation only — not done in real unsupervised analysis)
from sklearn.metrics import adjusted_rand_score
print("Adjusted Rand Index vs true labels:", adjusted_rand_score(y_iris, labels_km).round(4))


In [ ]:
# ── Silhouette analysis: choose best K ───────────────────────────────────────
sil_scores = []
for k in range(2, 10):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_km)
    sil_scores.append(silhouette_score(X_km, km.labels_))

plt.figure(figsize=(6,4))
plt.plot(range(2, 10), sil_scores, 'o-')
plt.xlabel("K"); plt.ylabel("Mean Silhouette Score")
plt.title("Silhouette Analysis"); plt.tight_layout(); plt.show()
print("Best K by silhouette:", np.argmax(sil_scores) + 2)


In [ ]:
# ── Hierarchical Clustering ──────────────────────────────────────────────────
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

X_hc = StandardScaler().fit_transform(X_iris)

# Compute linkage matrix (complete linkage + Euclidean distance)
Z = linkage(X_hc, method='complete', metric='euclidean')

plt.figure(figsize=(10,5))
dendrogram(Z,
           truncate_mode='lastp',  # show only last 20 merges (otherwise 150 leaves)
           p=20,
           leaf_rotation=90, leaf_font_size=10)
plt.title("Dendrogram — Complete Linkage"); plt.xlabel("Cluster"); plt.ylabel("Distance")
plt.tight_layout(); plt.show()


In [ ]:
# ── Cut dendrogram to get flat clusters ──────────────────────────────────────
labels_hc = fcluster(Z,
                      t=3,            # cut at 3 clusters
                      criterion='maxclust')

print("HC cluster sizes:", np.bincount(labels_hc))
print("HC Silhouette score:", silhouette_score(X_hc, labels_hc).round(4))

# ── Compare linkage methods ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, method in zip(axes, ['complete', 'single', 'average', 'ward']):
    Z_m = linkage(X_hc, method=method)
    dendrogram(Z_m, ax=ax, truncate_mode='lastp', p=12,
               no_labels=True, color_threshold=0)
    ax.set_title(method.capitalize())
plt.suptitle("Dendrogram comparison — linkage methods", y=1.02)
plt.tight_layout(); plt.show()
# Ward's linkage minimises within-cluster variance at each merge (most similar to K-means)


In [ ]:
# ── sklearn AgglomerativeClustering (for getting labels directly) ─────────────
hc = AgglomerativeClustering(n_clusters=3,
                               linkage='ward',
                               metric='euclidean')
labels_sk = hc.fit_predict(X_hc)
print("Ward's HC Silhouette:", silhouette_score(X_hc, labels_sk).round(4))


---
## 11 · Quick Reference — Model Skeleton

Every sklearn model follows the same 4-line pattern:

```python
from sklearn.XXX import YYY
model = YYY(hyperparams)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)          # class labels or regression values
y_prob = model.predict_proba(X_test)    # probabilities (classifiers only)
score  = model.score(X_test, y_test)    # R² (regression) or accuracy (classification)
```

| Method | Import | Key hyperparameters |
|--------|--------|---------------------|
| OLS | `sklearn.linear_model.LinearRegression` | — |
| Ridge | `sklearn.linear_model.Ridge` | `alpha` (= λ) |
| Lasso | `sklearn.linear_model.Lasso` | `alpha` (= λ) |
| Logistic Reg | `sklearn.linear_model.LogisticRegression` | `C` (= 1/λ), `max_iter` |
| LDA | `sklearn.discriminant_analysis.LinearDiscriminantAnalysis` | — |
| QDA | `sklearn.discriminant_analysis.QuadraticDiscriminantAnalysis` | — |
| KNN | `sklearn.neighbors.KNeighborsClassifier` | `n_neighbors` |
| Decision Tree | `sklearn.tree.DecisionTreeClassifier` | `max_depth`, `ccp_alpha` |
| Random Forest | `sklearn.ensemble.RandomForestClassifier` | `n_estimators`, `max_features` |
| Gradient Boost | `sklearn.ensemble.GradientBoostingClassifier` | `n_estimators`, `learning_rate`, `max_depth` |
| SVM (RBF) | `sklearn.svm.SVC` | `C`, `gamma`, `kernel` |
| Neural Net | `sklearn.neural_network.MLPClassifier` | `hidden_layer_sizes`, `alpha`, `learning_rate` |
| PCA | `sklearn.decomposition.PCA` | `n_components` |
| K-Means | `sklearn.cluster.KMeans` | `n_clusters`, `n_init` |
| Hierarchical | `sklearn.cluster.AgglomerativeClustering` | `n_clusters`, `linkage` |
